# 🧠📍 Smart Tagging

Metrica's **automatic** tactical annotation of the tactical video: phases of play and set
pieces, each one a time interval with a code and a set of tags.

This notebook covers loading it, what is in it, and the two things that will bite you if you
treat it as ground truth.

---

## Loading

The file is a **SportsCode XML**, the standard format in football performance analysis, so
[kloppy](https://kloppy.pysport.org/user-guide/loading-data/sportscode/) reads it directly.

`games.asset_paths()` builds the path, so nothing here spells a filename out and
`MTAH_DATA_DIR` keeps working if you store the bundle elsewhere.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

%pip install kloppy pandas numpy

import pandas as pd
from kloppy import sportscode

import games

path = games.asset_paths()['smart_tagging']
events = sportscode.load(path).to_df()
print('%d instances' % len(events))
events.head()

## What is in it

Each row is one tagged interval:

- **timestamp / end_timestamp** — start and end, as pandas timedeltas from the first frame
- **code** — the phase or set piece, upper case (`SET PIECES`, `BUILD UP`, …)
- **Team** — which side the phase belongs to
- **Half** — `1st Half` or `2nd Half`, as strings
- plus per-code tags: `Type`, `Side`, `Max Players in the box`, `Direction of ball entry`

Note the capital in `Team`: kloppy passes the tag groups through with the names the XML uses, so
every tag column is title case. The full vocabulary is in [DATA.md](../DATA.md).

In [ ]:
events['code'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

ev = events.copy()
ev['start_min'] = ev['timestamp'].dt.total_seconds() / 60
ev['dur_min'] = (ev['end_timestamp'] - ev['timestamp']).dt.total_seconds() / 60

codes = sorted(ev['code'].unique())
team_colour = {'Barcelona': 'tab:blue', 'Atletic Club': 'tab:red', 'N/A': 'tab:gray'}

fig, ax = plt.subplots(figsize=(14, 7))
for i, code in enumerate(codes):
    for _, row in ev[ev['code'] == code].iterrows():
        ax.barh(i, row['dur_min'], left=row['start_min'], height=0.6,
                color=team_colour.get(row['Team'], 'tab:gray'))

ax.set_yticks(range(len(codes)))
ax.set_yticklabels(codes)
ax.invert_yaxis()
ax.set_xlabel('minutes from the start of the video')
ax.set_title('Smart Tagging timeline (blue = Barcelona, red = Atletic Club, grey = N/A)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

KICKOFF_MIN = 11185 / 25 / 60       # first-half kick-off, about 7.46 minutes
WINDOW_START = KICKOFF_MIN
WINDOW_END = KICKOFF_MIN + 4        # first 4 minutes of play

ev = events.copy()
ev['start_min'] = ev['timestamp'].dt.total_seconds() / 60
ev['end_min'] = ev['end_timestamp'].dt.total_seconds() / 60
ev['dur_min'] = ev['end_min'] - ev['start_min']

# keep only tags that overlap the window
ev = ev[(ev['end_min'] >= WINDOW_START) & (ev['start_min'] <= WINDOW_END)]

codes = sorted(events['code'].unique())   # same row order for every window
team_colour = {'Barcelona': 'tab:blue', 'Atletic Club': 'tab:red', 'N/A': 'tab:gray'}

fig, ax = plt.subplots(figsize=(14, 7))
for i, code in enumerate(codes):
    for _, row in ev[ev['code'] == code].iterrows():
        ax.barh(i, row['dur_min'], left=row['start_min'], height=0.6,
                color=team_colour.get(row['Team'], 'tab:gray'))

ax.set_yticks(range(len(codes)))
ax.set_yticklabels(codes)
ax.invert_yaxis()
ax.set_xlim(WINDOW_START, WINDOW_END)
ax.set_xlabel('minutes from the start of the video')
ax.set_title('Smart Tagging, first 4 minutes of play (blue = Barcelona, red = Atletic Club, grey = N/A)')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ev = events.copy()
ev['start_s'] = ev['timestamp'].dt.total_seconds()
ev['end_s'] = ev['end_timestamp'].dt.total_seconds()

# 1. How many tags are active at each second of the match
t = np.arange(0, ev['end_s'].max(), 1.0)
active = np.zeros_like(t)
for _, row in ev.iterrows():
    active[(t >= row['start_s']) & (t < row['end_s'])] += 1

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(t / 60, active, step='post', alpha=0.6)
ax.set_xlabel('minutes from the start of the video')
ax.set_ylabel('tags active')
ax.set_title('Number of Smart Tagging phases active at each moment of the match')
ax.grid(axis='both', alpha=0.3)
plt.tight_layout()
plt.show()

print('largest number of tags active at once:', int(active.max()))

# 2. Every pair of tags that overlap in time
rows = ev.sort_values('start_s').reset_index(drop=True)
pairs = []
for i in range(len(rows)):
    for j in range(i + 1, len(rows)):
        if rows.loc[j, 'start_s'] >= rows.loc[i, 'end_s']:
            break   # rows are sorted by start, so no later row can overlap row i
        overlap_s = min(rows.loc[i, 'end_s'], rows.loc[j, 'end_s']) - rows.loc[j, 'start_s']
        pairs.append({
            'start_min': round(rows.loc[j, 'start_s'] / 60, 2),
            'code_a': rows.loc[i, 'code'], 'team_a': rows.loc[i, 'Team'],
            'code_b': rows.loc[j, 'code'], 'team_b': rows.loc[j, 'Team'],
            'overlap_s': round(overlap_s, 1),
        })

overlaps = pd.DataFrame(pairs)
print('overlapping pairs:', len(overlaps))

# Pairs where the two tags belong to different teams are the most informative
conflicts = overlaps[overlaps['team_a'] != overlaps['team_b']]
print('overlapping pairs with different teams:', len(conflicts))
conflicts.sort_values('overlap_s', ascending=False).head(20)

In [ ]:
import numpy as np
import pandas as pd

ev = events.reset_index(drop=True)
s = ev['timestamp'].dt.total_seconds().to_numpy()
e = ev['end_timestamp'].dt.total_seconds().to_numpy()
code = ev['code'].to_numpy()
team = ev['Team'].to_numpy()

# 1. For each tag, find the opposite-team tag whose time span overlaps it most
records = []
for i in range(len(ev)):
    inter = np.minimum(e[i], e) - np.maximum(s[i], s)
    union = np.maximum(e[i], e) - np.minimum(s[i], s)
    iou = np.where(union > 0, inter / union, 0.0)   # 1.0 = identical time span
    valid = (code != code[i]) & (team != team[i]) & (team != 'N/A') & (team[i] != 'N/A')
    if not valid.any():
        continue
    iou = np.where(valid, iou, -1.0)
    j = int(np.argmax(iou))
    records.append({
        'code_a': code[i], 'team_a': team[i],
        'code_b': code[j], 'team_b': team[j],
        'best_iou': round(iou[j], 2),
    })

best = pd.DataFrame(records)

# 2. For each pair of codes, the share of code_a tags that mirror code_b (IoU >= 0.7)
best['mirrored'] = best['best_iou'] >= 0.7
mirrored = (best[best['mirrored']]
            .groupby(['code_a', 'code_b']).size()
            .rename('mirrored_count').reset_index())
totals = best.groupby('code_a').size().rename('total_a').reset_index()
summary = mirrored.merge(totals, on='code_a')
summary['share'] = (summary['mirrored_count'] / summary['total_a']).round(2)

summary.sort_values('share', ascending=False).head(20)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# One side of each mirror pair. Keep the attacking side, because its Team is the team in possession.
DROP_MIRRORS = [
    'DEFENDING IN MIDDLE THIRD',     # mirror of PROGRESSION
    'DEFENSIVE TRANSITION',          # mirror of ATTACKING TRANSITION
    'DEFENDING IN ATTACKING THIRD',  # mirror of BUILD UP
    'DEFENDING IN DEFENSIVE THIRD',  # mirror of CREATING CHANCES
]

ev = events[~events['code'].isin(DROP_MIRRORS)].copy()
ev['start_min'] = ev['timestamp'].dt.total_seconds() / 60
ev['dur_min'] = (ev['end_timestamp'] - ev['timestamp']).dt.total_seconds() / 60

codes = sorted(ev['code'].unique())
team_colour = {'Barcelona': 'tab:blue', 'Atletic Club': 'tab:red', 'N/A': 'tab:gray'}

fig, ax = plt.subplots(figsize=(16, 7))
for i, code in enumerate(codes):
    for _, row in ev[ev['code'] == code].iterrows():
        ax.barh(i, row['dur_min'], left=row['start_min'], height=0.6,
                color=team_colour.get(row['Team'], 'tab:gray'))

ax.set_yticks(range(len(codes)))
ax.set_yticklabels(codes)
ax.invert_yaxis()
ax.set_xlabel('minutes from the start of the video')
ax.set_title('Smart Tagging, whole match, mirrored defending codes removed')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('tags kept:', len(ev), 'of', len(events))

In [ ]:
import os
import numpy as np
import pandas as pd
from kloppy import sportscode
import games

n = len(merged)
codes = merged['raw'].to_numpy()
paths = games.asset_paths()

# Dead-ball guess: set pieces and goals from Smart Tagging
tags = sportscode.load(paths['smart_tagging']).to_df()
restarts = tags[tags['code'].isin(['SET PIECES', 'GOALS'])]
restart_window = np.zeros(n, dtype=bool)
for _, r in restarts.iterrows():
    a = max(int(round(r['timestamp'].total_seconds() * 25)), 0)
    b = min(int(round(r['end_timestamp'].total_seconds() * 25)), n)
    restart_window[a:b] = True

# States: 0 = H, 1 = A, 2 = D
emit = np.zeros((n, 3))
emit[codes == 0] = [0.0, 1.0, 1.0]        # raw says home
emit[codes == 1] = [1.0, 0.0, 1.0]        # raw says away
emit[restart_window] = [2.0, 2.0, 0.0]    # inside a restart window: lean towards D

def make_trans(sw, td):
    # sw: cost of switching between H and A
    # td: cost of entering or leaving dead ball
    return np.array([[0.0, sw, td],
                     [sw, 0.0, td],
                     [td, td, 0.0]])

def viterbi(emit, trans):
    T = emit.shape[0]
    cost = emit[0].copy()
    back = np.zeros((T, 3), dtype=np.int8)
    for t in range(1, T):
        cand = cost[:, None] + trans
        back[t] = np.argmin(cand, axis=0)
        cost = cand[back[t], np.arange(3)] + emit[t]
    path = np.zeros(T, dtype=np.int8)
    path[-1] = np.argmin(cost)
    for t in range(T - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

# 1. Sweep the costs and look at the structure of each result
rows = []
for sw in [10.0, 20.0, 40.0]:
    for td in [5.0, 15.0, 30.0]:
        p = viterbi(emit, make_trans(sw, td))
        lab = np.array(['H', 'A', 'D'])[p]
        ha = lab[lab != 'D']
        rows.append({'switch': sw, 'to_dead': td,
                     'possessions': int((ha[1:] != ha[:-1]).sum()) + 1,
                     'D_share': round(float(np.mean(lab == 'D')), 3)})
print(pd.DataFrame(rows).to_string(index=False))

# 2. Choose the costs from the table: possessions in the hundreds, plausible D share
SWITCH = 20.0
TO_DEAD = 15.0

# 3. Final labels with the chosen costs
path = viterbi(emit, make_trans(SWITCH, TO_DEAD))
labels = np.array(['H', 'A', 'D'])[path]

print(pd.Series(labels).value_counts().to_string())
ha = labels[labels != 'D']
print('possessions (H or A runs):', int((ha[1:] != ha[:-1]).sum()) + 1)

# 4. Write and check
out = '../evaluation/results/my_hmm_v3.csv'
pd.DataFrame({'frame': np.arange(n), 'label': labels}).to_csv(out, index=False)
print('written:', out)

!python ../evaluation/validate_submission.py {out}
!python ../evaluation/run_evaluation.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kloppy import sportscode
import games

# Smart Tagging, with the mirrored defending codes removed
paths = games.asset_paths()
tags = sportscode.load(paths['smart_tagging']).to_df()

DROP_MIRRORS = ['DEFENDING IN MIDDLE THIRD', 'DEFENSIVE TRANSITION',
                'DEFENDING IN ATTACKING THIRD', 'DEFENDING IN DEFENSIVE THIRD']
ev = tags[~tags['code'].isin(DROP_MIRRORS)].copy()
ev['start_min'] = ev['timestamp'].dt.total_seconds() / 60
ev['dur_min'] = (ev['end_timestamp'] - ev['timestamp']).dt.total_seconds() / 60

team_colour = {'Barcelona': 'tab:blue', 'Atletic Club': 'tab:red', 'N/A': 'tab:gray'}

# The v3 labels, read from the file so the plot can't show another version by mistake
labels = pd.read_csv('../evaluation/results/my_hmm_v3.csv')['label'].to_numpy()
n = len(labels)
total_min = n / 25 / 60

# One colour per second for the label strip
step = 25
colour = {'H': (0.23, 0.47, 0.71), 'A': (0.77, 0.22, 0.19), 'D': (0.5, 0.5, 0.5)}
strip = np.array([colour[x] for x in labels[::step]])[None, :, :]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})

# Top: the tags
codes_plot = sorted(ev['code'].unique())
for i, c in enumerate(codes_plot):
    for _, r in ev[ev['code'] == c].iterrows():
        ax1.barh(i, r['dur_min'], left=r['start_min'], height=0.6,
                 color=team_colour.get(r['Team'], 'tab:gray'))
ax1.set_yticks(range(len(codes_plot)))
ax1.set_yticklabels(codes_plot)
ax1.invert_yaxis()
ax1.set_title('Smart Tagging (top) and my_hmm_v3 labels (bottom)')
ax1.grid(axis='x', alpha=0.3)

# Bottom: the labels
ax2.imshow(strip, aspect='auto', extent=[0, total_min, 0, 1])
ax2.set_yticks([])
ax2.set_xlabel('minutes from the start of the video')
ax2.set_title('Labels: blue = H, red = A, grey = D')

plt.tight_layout()
plt.show()

print('label counts:', pd.Series(labels).value_counts().to_dict())

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kloppy import metrica
import games

warnings.filterwarnings("ignore", message="All-NaN slice encountered")

PITCH_X, PITCH_Y = 105.3, 68.0   # metres
FPS = 25
NEAR_M = 3.0                     # "in contact" radius in metres
STILL_MS = 0.5                   # ball speed below this counts as still (m/s)
STILL_FRAMES = 3 * FPS           # still for at least 3 s

# 1. Load tracking (same files as notebook 03)
paths = games.asset_paths(source='tactical')
print("path keys:", list(paths.keys()))
meta_key = next(k for k in paths if 'meta' in k.lower())
raw_key = next(k for k in paths if ('raw' in k.lower() or 'track' in k.lower()) and k != meta_key)

dataset = metrica.load_tracking_epts(paths[meta_key], paths[raw_key])
full = dataset.to_df()
n = len(full)
frame = full['frame_id'].to_numpy() - 1          # frame_id counts from 1
assert (frame == np.arange(n)).all(), "frame index is not 0..n-1"

bx = full['ball_x'].to_numpy(float)
by = full['ball_y'].to_numpy(float)

# 2. Split players into home and away using the match metadata
ground_of = {}
for team in dataset.metadata.teams:
    ground = getattr(team.ground, 'value', str(team.ground)).lower()
    side = 'H' if 'home' in ground else 'A'
    for player in team.players:
        ground_of[str(player.player_id)] = side

player_prefixes = sorted({c[:-2] for c in full.columns
                          if c.endswith('_x') and not c.startswith('ball')})
home = [p for p in player_prefixes if ground_of.get(p) == 'H']
away = [p for p in player_prefixes if ground_of.get(p) == 'A']
unassigned = [p for p in player_prefixes if p not in ground_of]

print(f"player columns: {len(player_prefixes)}  home: {len(home)}  "
      f"away: {len(away)}  unassigned: {len(unassigned)}")
if unassigned:
    print("unassigned ids (ignored):", unassigned)
if not home or not away:
    raise ValueError("No home or away players found in metadata. Print dataset.metadata.teams to check.")

# 3. Distance from the ball to every player, in metres
def distances(team):
    d = np.full((n, len(team)), np.nan)
    for j, p in enumerate(team):
        x = full[p + '_x'].to_numpy(float) * PITCH_X
        y = full[p + '_y'].to_numpy(float) * PITCH_Y
        d[:, j] = np.hypot(x - bx * PITCH_X, y - by * PITCH_Y)
    return d

dH = distances(home)
dA = distances(away)
d_home = np.nanmin(dH, axis=1)     # nearest home player (m)
d_away = np.nanmin(dA, axis=1)     # nearest away player (m)
n_home_near = np.nansum(dH < NEAR_M, axis=1)
n_away_near = np.nansum(dA < NEAR_M, axis=1)

# 4. Ball features
missing = np.isnan(bx) | np.isnan(by)
out_of_play = (~missing) & ((bx < 0) | (bx > 1) | (by < 0) | (by > 1))

step = np.hypot(np.diff(bx) * PITCH_X, np.diff(by) * PITCH_Y) * FPS
speed = np.r_[np.nan, step]                      # m/s, NaN where ball is missing

def run_length(mask):
    """For each frame in a True run, the length of that run; 0 elsewhere."""
    ids = np.cumsum(np.r_[True, mask[1:] != mask[:-1]])
    sizes = pd.Series(mask).groupby(ids).transform('size').to_numpy()
    return np.where(mask, sizes, 0)

missing_run = run_length(missing)
still = (speed < STILL_MS) & ~missing
still_3s = pd.Series(still).rolling(STILL_FRAMES, center=True, min_periods=STILL_FRAMES) \
                           .min().fillna(0).astype(bool).to_numpy()

# 5. Soft possession score: near 1 = home closer to ball, near 0 = away closer
p_home = d_away / (d_home + d_away)
p_home[missing] = np.nan

feat = pd.DataFrame({
    'frame': frame,
    'd_home_m': d_home, 'd_away_m': d_away,
    'p_home': p_home,
    'n_home_near': n_home_near, 'n_away_near': n_away_near,
    'contest': (n_home_near > 0) & (n_away_near > 0),
    'ball_speed_ms': speed,
    'ball_missing': missing,
    'missing_run_frames': missing_run,
    'ball_out': out_of_play,
    'ball_still_3s': still_3s,
})

# 6. Save features locally (derived from match data: do not commit)
os.makedirs('../work', exist_ok=True)
feat.to_csv('../work/features_v1.csv', index=False)

# 7. Sanity summary
def share(mask): return f"{mask.mean()*100:5.1f}%"
print("\n--- shares of all frames ---")
print("ball missing         ", share(feat['ball_missing'].to_numpy()))
print("ball out of play     ", share(feat['ball_out'].to_numpy()))
print("ball still 3 s+      ", share(feat['ball_still_3s'].to_numpy()))
print("contest (both near)  ", share(feat['contest'].to_numpy()))
print("p_home > 0.7         ", f"{np.nanmean(feat['p_home'].to_numpy() > 0.7)*100:5.1f}%")

out_spells = pd.Series(out_of_play).groupby(
    np.cumsum(np.r_[True, out_of_play[1:] != out_of_play[:-1]])).agg(['first', 'size'])
out_spells = out_spells[out_spells['first']]['size'] / FPS
print(f"\nout-of-play spells: {len(out_spells)}  "
      f"median {out_spells.median():.1f} s  longest {out_spells.max():.0f} s")

# 8. Plot the first 4 minutes to eyeball the features
end = 4 * 60 * FPS
t = np.arange(end) / FPS / 60
fig, ax = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
ax[0].plot(t, feat['p_home'].to_numpy()[:end]); ax[0].set_ylabel('p_home')
ax[1].plot(t, feat['ball_speed_ms'].to_numpy()[:end]); ax[1].set_ylabel('ball m/s')
ax[2].plot(t, feat['ball_out'].to_numpy()[:end].astype(int), label='out')
ax[2].plot(t, feat['ball_still_3s'].to_numpy()[:end].astype(int) * 0.5, label='still 3s')
ax[2].plot(t, feat['ball_missing'].to_numpy()[:end].astype(int) * 0.25, label='missing')
ax[2].legend(loc='upper right'); ax[2].set_ylabel('flags')
ax[3].plot(t, feat['contest'].to_numpy()[:end].astype(int)); ax[3].set_ylabel('contest')
ax[3].set_xlabel('minutes')
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FPS = 25
feat = pd.read_csv('../work/features_v1.csv')

# 1. Missing-ball runs across the whole match
grp = (feat['ball_missing'] != feat['ball_missing'].shift()).cumsum()
runs = feat.groupby(grp)['ball_missing'].agg(['first', 'size'])
miss_runs = runs[runs['first']]['size'] / FPS
print(f"missing-ball runs: {len(miss_runs)}")
print(f"median {miss_runs.median():.1f} s  90th pct {miss_runs.quantile(0.9):.1f} s  longest {miss_runs.max():.0f} s")
print(f"share of missing frames in runs over 3 s: "
      f"{miss_runs[miss_runs > 3].sum() / miss_runs.sum() * 100:.1f}%")

# 2. Plot the first graded four minutes (frame 11185 onward)
start = 11185
end = start + 4 * 60 * FPS
w = feat.iloc[start:end]
t = (w['frame'].to_numpy() - start) / FPS / 60 + start / FPS / 60

fig, ax = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
ax[0].plot(t, w['p_home'].to_numpy()); ax[0].set_ylabel('p_home')
ax[1].plot(t, w['ball_speed_ms'].to_numpy()); ax[1].set_ylabel('ball m/s')
ax[2].plot(t, w['ball_out'].astype(int).to_numpy(), label='out')
ax[2].plot(t, w['ball_missing'].astype(int).to_numpy() * 0.5, label='missing (x0.5)')
ax[2].plot(t, w['contest'].astype(int).to_numpy() * 0.75, label='contest (x0.75)')
ax[2].legend(loc='upper right'); ax[2].set_ylabel('flags'); ax[2].set_xlabel('minutes')
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import pandas as pd

FPS = 25
feat = pd.read_csv('../work/features_v1.csv')

# 1. Missing-ball runs with start time and length
grp = (feat['ball_missing'] != feat['ball_missing'].shift()).cumsum()
runs = feat.groupby(grp).agg(
    missing=('ball_missing', 'first'),
    start=('frame', 'min'),
    size=('frame', 'size'),
)
miss = runs[runs['missing']].copy()
miss['start_min'] = miss['start'] / FPS / 60
miss['len_s'] = miss['size'] / FPS

# 2. Is each run inside a graded window?
graded = (((miss['start'] >= 11185) & (miss['start'] <= 90651)) |
          ((miss['start'] >= 110808) & (miss['start'] <= 190197)))
miss['graded'] = graded

print("longest missing-ball runs:")
print(miss.sort_values('len_s', ascending=False).head(15)[['start_min', 'len_s', 'graded']]
      .round(2).to_string(index=False))

print(f"\ngraded runs: {graded.sum()}  "
      f"graded missing seconds: {miss.loc[graded, 'len_s'].sum():.0f} s  "
      f"ungraded missing seconds: {miss.loc[~graded, 'len_s'].sum():.0f} s")

# 3. Ball speed: raw vs median-filtered
speed = feat['ball_speed_ms']
speed_smooth = speed.rolling(5, center=True, min_periods=1).median()
print("\nball speed m/s (raw):    ", speed.quantile([0.5, 0.99]).round(1).tolist(),
      " max", round(speed.max(), 1))
print("ball speed m/s (smoothed):", speed_smooth.quantile([0.5, 0.99]).round(1).tolist(),
      " max", round(speed_smooth.max(), 1))

In [ ]:
import pandas as pd
sb = pd.read_csv('../evaluation/scoreboard.csv')
print(sb[['submission', 'spell_f1_mean', 'poss', 'dur']].to_string())

### Codes are upper case, tag values are title case

`SET PIECES` and `Kick Off (Start)`. Match case-insensitively unless you enjoy silent empty
results.

In [ ]:
set_pieces = events[events['code'].str.lower() == 'set pieces']
set_pieces['Type'].value_counts()

## Converting to video frames

The timestamps are **seconds from the first frame of the tactical video**, so a video frame —
and therefore a submission row — is just `round(seconds * 25)`. There is no offset to discover.

`starter/load_smart_tagging.py` does this without kloppy, and re-checks the time base against
both kickoffs every time it runs.

In [ ]:
FPS = games.GAMES[games.GAME]['fps']

frames = pd.DataFrame({
    'code': events['code'],
    'Team': events['Team'],
    'start_frame': (events['timestamp'].dt.total_seconds() * FPS).round().astype(int),
    'end_frame': (events['end_timestamp'].dt.total_seconds() * FPS).round().astype(int),
})
frames.head()

## Two things to be careful about

### 1. `Team` is not always a team

36 instances are tagged `N/A` — including **every throw-in**, which are exactly the restarts
where knowing the side would be worth the most. Handle it explicitly; do not let it fall
through to a default.

In [ ]:
events['Team'].value_counts(dropna=False)

### 2. An interval is a window, not an instant

A set piece is tagged as a span of a few seconds *around* the restart, not at the frame the ball
is struck. The two `Kick Off (Start)` tags below both **contain** the real kickoff rather than
starting on it — they open about 3.5 seconds early.

So `start_frame` is not the restart frame, and the same imprecision applies to every other
interval in the file. Use a tag to say *roughly when* something happened, and the tracking to say
exactly when.

In [ ]:
kickoffs = set_pieces[set_pieces['Type'].str.lower() == 'kick off (start)'].copy()
kickoffs['start_frame'] = (kickoffs['timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['end_frame'] = (kickoffs['end_timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['real_kickoff'] = [11185, 113742]          # from the ground truth
kickoffs['contains_it'] = ((kickoffs['start_frame'] <= kickoffs['real_kickoff'])
                           & (kickoffs['real_kickoff'] <= kickoffs['end_frame']))

kickoffs[['Half', 'start_frame', 'end_frame', 'real_kickoff', 'contains_it']]

---

## Why this is worth using anyway

Every phase carries a **team** tag, and every possession has a team. Every set piece is a
restart, and every restart ends a dead-ball stretch. The correspondence is not exact — that is
the challenge — but it is a strong prior, for free, on both halves of the label you are asked
to produce.

Just remember the intervals are *phases*, not possessions: a Build Up can span one possession,
stop short of one, or cover two.

**Next:** [`02_tracking_atd.ipynb`](02_tracking_atd.ipynb) for the tracking data, then
[`03_first_submission.ipynb`](03_first_submission.ipynb) to build and score a submission.